In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
file_path = "sample.txt"
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()
print(f"Loaded {len(documents)} document(s)")

Loaded 1 document(s)


In [ ]:
splitter = CharacterTextSplitter(
 separator="\n",
 chunk_size=500,
 chunk_overlap=50
)
split_docs = splitter.split_documents(documents)
print(f"Total chunks created: {len(split_docs)}")

Total chunks created: 27


In [ ]:
embeddings = HuggingFaceEmbeddings(
 model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = FAISS.from_documents(
 documents=split_docs,
 embedding=embeddings
)


In [ ]:
query = "What is this document about?"
results = vectorstore.similarity_search(query, k=3)


In [ ]:
for i, doc in enumerate(results, start=1):
 print(f"\nResult {i}")
 print("-" * 50)
 print(doc.page_content)
 print("Metadata:", doc.metadata)


Result 1
--------------------------------------------------
(B) The Lessee wishes to lease the Property from the Lessor for use as office space.
(C) The Parties wish to establish the terms and conditions of the Property lease through this Agreement.
THE FOLLOWING IS AGREED:
1 Definitions and interpretation
This Agreement and the capitalized terms mentioned therein shall be interpreted in accordance with the guidelines and definitions set out in Appendix 1 to this Agreement.
2 Object of lease
Metadata: {'source': 'sample.txt'}

Result 2
--------------------------------------------------
11.3 Maintenance and repair works not included in Article 11.1 shall be borne by the Lessor. The Lessee must immediately notify the Lessor in writing of damage or deterioration falling outside their repair obligation. The Lessee must tolerate such maintenance and repair works by the Lessor, as well as urgent repairs that cannot be postponed and that the Lessor has carried out, even if these works take m